## 2024-09-24: CNN Exploraiton

### Authors
* Nicole Tin (nicole@velexi.com)


### Overview
This Jupyter notebook is intended to explore

* various packages/architectures for constructing a CNN to use on images

### Key Results

The key results of this experiment are ...

### Future Ideas
* benchmark with professional dermatologists estimate (+-10 yrs)
  * professional skin attribute assessment
* inter-group variability
* ML nor doctors may not be good at identifying biological age
* scientifically, there may be more group variability
* one person in training/test split as L/R hands
  * should it be strictly L or R?
  * train only on L, test on R, would outcomes be perfect? high error? 


### EDA ideas (age vs sun exposure)
* L vs. R
* inter age

### Experiment Parameters

In [2]:
# --- Experiment parameters

# Name of experiment. Used for MLflow experiment name, output files, etc.
experiment_name = "example-experiment"

# Paths
src_dir = '/Users/nicole/Documents/DermaML_local/hawkeye-hands-2024-07-29'
image_dir = '/processed_images/'
csv_file = '/metadata.csv'


# ------ Algorithm parameters

# train/test split
split = 0.7
epochs = 10
learning_rate = 1e3

### Preparations

In [3]:
# --- Imports

# Internal library
from dermaml.data import read_local
from dermaml.CNN.softmax import Softmax
from dermaml.CNN.maxpool import MaxPool2
from dermaml.CNN.conv import Conv3x3

# Standard library
import random 
import math
from datetime import datetime
# External packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Entropy
random.seed(42)
np.random.seed(42)

np.set_printoptions(precision=3, suppress=True)

In [25]:
metadata = pd.read_csv(src_dir+ csv_file)
metadata.loc[:, 'Age'] = 2024-metadata['birth_year']
valid_image_fnames_df = pd.DataFrame(metadata.set_index('Age').loc[:, ['right_hand_image_file', 'left_hand_image_file']].stack()).reset_index()
valid_image_fnames_df.columns = ['Age', 'handedness', 'filename']
valid_image_fnames_df.loc[:,'filename'] = valid_image_fnames_df['filename'].apply(lambda x: x[:-5])
valid_image_fnames = valid_image_fnames_df['filename'].to_numpy()
y = valid_image_fnames_df.loc[:, 'Age'].to_numpy()

In [18]:
hawkeye_filenames, hawkeye_hands_images = read_local(src_dir+image_dir)

image_count = len(hawkeye_hands_images)

  0%|          | 0/577 [00:00<?, ?it/s]

100%|██████████| 577/577 [03:00<00:00,  3.20it/s]


In [20]:
# image_data = hawkeye_hands_images.reshape(
#     hawkeye_hands_images.shape + (1, ))
import cv2 as cv

def cnn_conformity(im):
    if len(im.shape)==3:
        bw_im = cv.cvtColor(im, cv.COLOR_BGR2GRAY)
    else:
        print('image is not 3 channel')
        bw_im = im
    smaller_bw_im = cv.resize(bw_im, (512, 512), 
               interpolation = cv.INTER_AREA)
    return smaller_bw_im

hh_images = [cnn_conformity(im) for im in hawkeye_hands_images]

train_size = math.floor(image_count * split)
# train_dataset = image_data[:train_size]
# test_dataset = image_data[train_size:]
train_images = np.array(hh_images[:train_size])
test_images = np.array(hh_images[train_size:])

# round labels to neearest fifth
train_labels = np.around(y[:train_size]/5, decimals=0)*5
test_labels = np.around(y[train_size:]/5, decimals=0)*5

test_results = {}

### Perform computational experiment

In [22]:
conv = Conv3x3(8)                  # 512x512x1 -> 510x510x8 ; padding decreases w,h by 2
pool = MaxPool2()                  # 510x510x8 -> 255x255x8
softmax = Softmax(2555 * 2555 * 8, 20) # 255x255x8 -> 20

In [ ]:
# self.weights = np.random.randn(input_len, nodes) / input_len
# self.biases = np.zeros(nodes)

In [23]:
def forward(image, label):
  '''
  Completes a forward pass of the CNN and calculates the accuracy and
  cross-entropy loss.
  - image is a 2d numpy array
  - label is a digit
  '''
  # We transform the image from [0, 255] to [-0.5, 0.5] to make it easier
  # to work with. This is standard practice.
  out = conv.forward((image / 255) - 0.5)
  out = pool.forward(out)
  out = softmax.forward(out)

  # Calculate cross-entropy loss and accuracy. np.log() is the natural log.
  loss = -np.log(out[label])
  acc = 1 if np.argmax(out) == label else 0

  return out, loss, acc

def train(im, label, lr=.005):
  '''
  Completes a full training step on the given image and label.
  Returns the cross-entropy loss and accuracy.
  - image is a 2d numpy array
  - label is a digit
  - lr is the learning rate
  '''
  # Forward
  out, loss, acc = forward(im, label)

  # Calculate initial gradient
  gradient = np.zeros(10)
  gradient[label] = -1 / out[label]

  # Backprop
  gradient = softmax.backprop(gradient, lr)
  gradient = pool.backprop(gradient)
  gradient = conv.backprop(gradient, lr)

  return loss, acc

In [24]:
train(train_images[0], train_labels[0])

ValueError: shapes (520200,) and (52224200,20) not aligned: 520200 (dim 0) != 52224200 (dim 0)

In [8]:
# -- EDITED code

# Train the CNN for 3 epochs
for epoch in range(3):
  print('--- Epoch %d ---' % (epoch + 1))

  # Shuffle the training data
  permutation = np.random.permutation(len(train_images))
  train_images = train_images[permutation]
  train_labels = train_labels[permutation]

  # Train!
  loss = 0
  num_correct = 0
  for i, (im, label) in enumerate(zip(train_images, train_labels)):
    if i % 100 == 99:
      print(
        '[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
        (i + 1, loss / 100, num_correct)
      )
      loss = 0
      num_correct = 0

    l, acc = train(im, label)
    loss += l
    num_correct += acc

# Test the CNN
print('\n--- Testing the CNN ---')
loss = 0
num_correct = 0
for im, label in zip(test_images, test_labels):
  _, l, acc = forward(im, label)
  loss += l
  num_correct += acc

num_tests = len(test_images)
print('Test Loss:', loss / num_tests)
print('Test Accuracy:', num_correct / num_tests)

--- Epoch 1 ---


ValueError: shapes (520200,) and (1352,10) not aligned: 520200 (dim 0) != 1352 (dim 0)

In [13]:
# -- original code

# Train the CNN for 3 epochs
for epoch in range(3):
  print('--- Epoch %d ---' % (epoch + 1))

  # Shuffle the training data
  permutation = np.random.permutation(len(train_images))
  train_images = train_images[permutation]
  train_labels = train_labels[permutation]

  # Train!
  loss = 0
  num_correct = 0
  for i, (im, label) in enumerate(zip(train_images, train_labels)):
    if i % 100 == 99:
      print(
        '[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
        (i + 1, loss / 100, num_correct)
      )
      loss = 0
      num_correct = 0

    l, acc = train(im, label)
    loss += l
    num_correct += acc

# Test the CNN
print('\n--- Testing the CNN ---')
loss = 0
num_correct = 0
for im, label in zip(test_images, test_labels):
  _, l, acc = forward(im, label)
  loss += l
  num_correct += acc

num_tests = len(test_images)
print('Test Loss:', loss / num_tests)
print('Test Accuracy:', num_correct / num_tests)

--- Epoch 1 ---


TypeError: only integer scalar arrays can be converted to a scalar index